# 1. Import Libraries

In [88]:
!pip install transformers pandas torch
!pip install 'accelerate>=1.1.0'
!pip install scikit-learn
!pip install datasets


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [89]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import accelerate
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
import numpy as np
from sklearn.metrics import classification_report

# 2. Import Dataset and Train-test Split

In [90]:
df = pd.read_csv("../dataset/dataset_absa.csv")

labels = ["Negatif", "Positif"]

def encode_sentiment(dataframe):
    dataframe["labels"] = dataframe["sentiment"].apply(labels.index)
    return dataframe

def decode_sentiment(dataframe):
    dataframe["sentiment_decoded"] = dataframe["labels"].apply(lambda x: labels[x])
    return dataframe

df = encode_sentiment(df)


total_reviews = df["review_id"].max()

train_percentage = 0.8

max_id_train = total_reviews * train_percentage

train_df = df[df["review_id"] <= max_id_train]
test_df = df[df["review_id"] > max_id_train]

train_df

,review_id,review,aspect,sentiment,labels
0,1,Mata uang Asia 'rontok' berjamaah pada perdaga...,Mata uang Asia,Negatif,0
1,1,Mata uang Asia 'rontok' berjamaah pada perdaga...,Won Korea Selatan,Negatif,0
2,1,Mata uang Asia 'rontok' berjamaah pada perdaga...,Baht Thailand,Negatif,0
3,1,Mata uang Asia 'rontok' berjamaah pada perdaga...,Ringgit Malaysia,Negatif,0
4,1,Mata uang Asia 'rontok' berjamaah pada perdaga...,Dolar Singapura,Negatif,0
...,...,...,...,...,...
379,8,Kebijakan tarif impor baru yang ditetapkan Pre...,kesempatan emas,Positif,1
380,8,Kebijakan tarif impor baru yang ditetapkan Pre...,berbagai saham dengan fundamental yang baik,Positif,1
381,8,Kebijakan tarif impor baru yang ditetapkan Pre...,tekanan di pasar saham,Negatif,0
382,8,Kebijakan tarif impor baru yang ditetapkan Pre...,valuasi saham menjadi sangat murah,Positif,1


# 3. Import IndoBERT

In [91]:
model_name = "indobenchmark/indobert-base-p2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28850.25it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 4. Fine Tune Model

In [92]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize(examples):
    return tokenizer(
        examples["review"],
        examples["aspect"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset

Map: 100%|██████████| 46/46 [00:00<00:00, 1864.89 examples/s]


Dataset({
    features: ['review_id', 'review', 'aspect', 'sentiment', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 384
})

In [93]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]


TrainOutput(global_step=120, training_loss=0.4335107167561849, metrics={'train_runtime': 45.9483, 'train_samples_per_second': 41.786, 'train_steps_per_second': 2.612, 'total_flos': 126293306572800.0, 'train_loss': 0.4335107167561849, 'epoch': 5.0})

# 5. Predict

In [ ]:
predictions = trainer.predict(test_dataset)
pred_ids = np.argmax(predictions.predictions, axis=1)

predictions_df = pd.DataFrame({
    "prediction": [labels[i] for i in pred_ids]
})

predictions_df["actuals"] = test_df.reset_index(drop=True)["sentiment"]
predictions_df["review_id"] = test_df.reset_index(drop=True)["review_id"]
predictions_df["sentiment"] = test_df.reset_index(drop=True)["sentiment"]
predictions_df

,prediction,actuals
0,Negatif,Positif
1,Positif,Positif
2,Negatif,Positif
3,Positif,Positif
4,Positif,Positif
5,Negatif,Negatif
6,Positif,Positif
7,Positif,Positif
8,Positif,Positif
9,Positif,Positif


# 6. Evaluate

In [95]:
print(
    classification_report(
        test_df["labels"],
        pred_ids,
        target_names=["Negatif", "Positif"]
    )
)

              precision    recall  f1-score   support

     Negatif       0.58      0.88      0.70        16
     Positif       0.91      0.67      0.77        30

    accuracy                           0.74        46
   macro avg       0.75      0.77      0.73        46
weighted avg       0.80      0.74      0.75        46



# 